# Research snapshot audit — 2026-09-14
## Summary
Two saved feature matrices share 105 trial IDs but disagree on 46 strict labels. In the 638-row snapshot, only three temporal-test trials have known targets unseen in training. These are snapshot diagnostics, not new model-training results.

## Context and methods
This companion verifies the saved audit and exposes the implementation. Exact table values are more useful here than charts. Sources are local snapshots; external trial outcomes were not adjudicated. Unknown entities are separate from unseen entities. Confidence intervals condition on saved predictions, not retraining.

### Execution note
Code cells were executed sequentially in the project Python environment and stdout saved. Jupyter/nbformat/nbclient are unavailable; no Jupyter kernel or notebook viewer validation was performed. A Python-generated notebook structure was checked locally. No dependencies were installed.


In [1]:
from pathlib import Path
import hashlib
import json
import sys
import pandas as pd
root = Path.cwd()
if not (root / "audit_research_snapshot.py").exists():
    root = root.parent
assert (root / "audit_research_snapshot.py").exists(), "Start in the repository root or docs directory"
sys.path.insert(0, str(root))
from audit_research_snapshot import overlap_rows, bootstrap_auc
out = root / "mechanistic_engine_output/research_audit_2026-09-14"
manifest = json.loads((out / "manifest.json").read_text())
for source in manifest["sources"]:
    folder = "reports" if source["file"] == "deep_predictions.csv" else "data"
    path = root / "mechanistic_engine_output" / folder / source["file"]
    assert hashlib.sha256(path.read_bytes()).hexdigest() == source["sha256"]
print("All three source hashes match the audit.")


All three source hashes match the audit.


## Data and source reconciliation

In [2]:
left = pd.read_csv(root / "mechanistic_engine_output/data/classification_matrix.csv")
right = pd.read_csv(root / "mechanistic_engine_output/data/full_feature_matrix.csv")
a = left.set_index("nct_id"); b = right.set_index("nct_id")
shared = a.index.intersection(b.index)
disagreements = (a.loc[shared, "label_strict"] != b.loc[shared, "label_strict"]).sum()
assert len(shared) == 105 and disagreements == 46
print(f"Classification rows: {len(left)}; full rows: {len(right)}")
print(f"Shared trials: {len(shared)}; strict label disagreements: {disagreements}")
print(pd.read_csv(out / "snapshot_comparison.csv").to_string(index=False))


Classification rows: 638; full rows: 1038
Shared trials: 105; strict label disagreements: 46
           field  common_trials  disagreements
    label_strict            105             46
  label_balanced            105             46
label_permissive            105             46
  canonical_drug            105             17
  primary_target            105             15
         disease            105             37
      start_year            105              0


## Results: temporal overlap
Counts refer to trial rows, not unique entities. Mapping changes across snapshots remain unresolved.

In [3]:
overlap = pd.DataFrame(overlap_rows(left, "label_strict"))
assert ((overlap.test_seen + overlap.test_unseen + overlap.test_unknown) == overlap.n_test).all()
print(overlap[["entity", "n_train", "n_test", "test_seen", "test_unseen", "test_unknown", "unseen_positive", "unseen_negative"]].to_string(index=False))


           entity  n_train  n_test  test_seen  test_unseen  test_unknown  unseen_positive  unseen_negative
   canonical_drug      472     166        139           16            11                5               11
   primary_target      472     166        152            3            11                2                1
drug_disease_pair      472     166         73           82            11               18               64


## Results: uncertainty on existing predictions
Trial-level percentile bootstrap, 2,000 replicates, seed 42. Model-selection and training uncertainty are not included.

In [4]:
pred = pd.read_csv(root / "mechanistic_engine_output/reports/deep_predictions.csv")
ci = bootstrap_auc(pred.label, pred.biology_mlp_prob, repeats=2000)
saved = pd.read_csv(out / "deep_auc_intervals.csv")
row = saved[(saved.model_probability == "biology_mlp_prob") & (saved.bootstrap_unit == "trial")].iloc[0]
for key in ["auc", "ci_low", "ci_high"]:
    assert abs(ci[key] - row[key]) < 1e-12
comparison = pd.read_csv(root / "mechanistic_engine_output/reports/deep_model_comparison.csv")
expected_auc = comparison.loc[comparison.model == "BiologyOnlyMLP", "roc_auc"].iloc[0]
assert abs(ci["auc"] - expected_auc) < 1e-6
print(saved[saved.bootstrap_unit == "trial"].to_string(index=False))


model_probability bootstrap_unit   n  positives  repeats      auc   ci_low  ci_high  valid_replicates
 biology_mlp_prob          trial 419        141     2000 0.529848 0.468470 0.592146              2000
  entity_mlp_prob          trial 419        141     2000 0.477818 0.420601 0.538122              2000
         mlp_prob          trial 419        141     2000 0.477524 0.418854 0.536076              2000
 transformer_prob          trial 419        141     2000 0.439385 0.383686 0.498635              2000


## Results: review queue
150 trials await review; no reviewer labels have been filled. The deliberately stratified queue cannot estimate population error rates without sampling weights.

In [5]:
queue = pd.read_csv(out / "label_review_queue.csv")
assert len(queue) == 150 and queue.nct_id.is_unique
assert queue.endpoint_met_yes_no_unclear.isna().all()
assert not any(column.startswith("label_") for column in queue)
print("150 unique trials; reviewer outcomes blank; heuristic labels stored separately.")
print(queue[["nct_id", "status", "start_year"]].head(5).to_string(index=False))


150 unique trials; reviewer outcomes blank; heuristic labels stored separately.
     nct_id    status  start_year
NCT00003270 COMPLETED      1997.0
NCT00003631 COMPLETED      1998.0
NCT00004092 COMPLETED      1999.0
NCT00039195 COMPLETED      2006.0
NCT00045110 COMPLETED      2002.0


## Reproduce the complete audit
Use a previously unused destination; the script refuses to overwrite existing output:

```bash
.venv/bin/python audit_research_snapshot.py --output-dir mechanistic_engine_output/research_audit_repeat
.venv/bin/python -m unittest discover -s tests -p 'test_research_snapshot_audit.py' -v
```

The full implementation is `audit_research_snapshot.py`, including source hashing, label provenance counts, unknown-entity handling, review sampling, and cluster bootstrap.

## Takeaways
Agree on one cohort and outcome definition before comparing models. Review conflicting labels. Treat unseen-target performance as exploratory because of the very small sample. September 7–13 meeting notes were not available; the discussion outline remains provisional.
